3. 记忆封装
   - Memory：这里不是物理内存，从文本的角度，可以理解为“上文”、“历史记录”或者说“记忆力”的管理
4. 架构封装
   - Chain：实现一个功能或者一系列顺序功能组合
   - Agent：根据用户输入，自动规划执行步骤，自动选择每步需要的工具，最终完成用户指定的功能
     - Tools：调用外部功能的函数，例如：调 google 搜索、文件 I/O、Linux Shell 等等
     - Toolkits：操作某软件的一组工具集，例如：操作 DB、操作 Gmail 等等
5. Callbacks

In [1]:
#local env -> C:\Users\hh\AppData\Roaming\Python\venvs

<h4>3.记忆封装 - Memory</h4>

In [3]:
from langchain.memory import ConversationBufferMemory

history = ConversationBufferMemory()
history.save_context({"input":"electronic是做什么的？"},{"output":"Electron是一个使用JavaScript、HTML和CSS构建桌面应用程序的框架"})
print(history.load_memory_variables({}))

history.save_context({"input":"给我一个它的教程链接？"},{"output":"https://www.electronjs.org/zh/docs/latest/tutorial/quick-start"})
print(history.load_memory_variables({}))

{'history': 'Human: electronic是做什么的？\nAI: Electron是一个使用JavaScript、HTML和CSS构建桌面应用程序的框架'}
{'history': 'Human: electronic是做什么的？\nAI: Electron是一个使用JavaScript、HTML和CSS构建桌面应用程序的框架\nHuman: 给我一个它的教程链接？\nAI: https://www.electronjs.org/zh/docs/latest/tutorial/quick-start'}


In [6]:
#Message格式
from langchain.memory import ChatMessageHistory
history = ChatMessageHistory()
history.add_user_message("electronic是做什么的？")
history.add_ai_message("Electron是一个使用JavaScript、HTML和CSS构建桌面应用程序的框架")
print(history)

messages=[HumanMessage(content='electronic是做什么的？', additional_kwargs={}, example=False), AIMessage(content='Electron是一个使用JavaScript、HTML和CSS构建桌面应用程序的框架', additional_kwargs={}, example=False)]


In [1]:
#保留指定window的历史会话信息
from langchain.memory import ConversationBufferWindowMemory

window = ConversationBufferWindowMemory(k=8) #k=8表示保留最近8轮的历史会话信息
window.save_context({"input": "第一轮问"}, {"output": "第一轮答"})
window.save_context({"input": "第二轮问"}, {"output": "第二轮答"})
window.save_context({"input": "第三轮问"}, {"output": "第三轮答"})
window.save_context({"input": "第四轮问"}, {"output": "第四轮答"})
window.save_context({"input": "第五轮问"}, {"output": "第五轮答"})
window.save_context({"input": "第六轮问"}, {"output": "第六轮答"})
window.save_context({"input": "第七轮问"}, {"output": "第七轮答"})
window.save_context({"input": "第八轮问"}, {"output": "第八轮答"})
window.save_context({"input": "第九轮问"}, {"output": "第九轮答"})
window.save_context({"input": "第十轮问"}, {"output": "第十轮答"})
print(window.load_memory_variables({}))


{'history': 'Human: 第三轮问\nAI: 第三轮答\nHuman: 第四轮问\nAI: 第四轮答\nHuman: 第五轮问\nAI: 第五轮答\nHuman: 第六轮问\nAI: 第六轮答\nHuman: 第七轮问\nAI: 第七轮答\nHuman: 第八轮问\nAI: 第八轮答\nHuman: 第九轮问\nAI: 第九轮答\nHuman: 第十轮问\nAI: 第十轮答'}


<h4>3.2 自动对历史信息做摘要：ConversationSummaryMemory</h4>

In [1]:
import warnings
warnings.filterwarnings("ignore")
from langchain.llms import OpenAI
from langchain.chat_models import ChatOpenAI
import os
from dotenv import load_dotenv
load_dotenv()
api_key = os.environ["OPENAI_B_API_KEY"]
api_base = os.environ["OPENAI_B_API_BASE"]

In [2]:
from langchain.memory import ConversationSummaryMemory
from langchain.llms import OpenAI

memory = ConversationSummaryMemory(
    llm=OpenAI(temperature=0,openai_api_base=api_base ,openai_api_key=api_key,verbose=True),
    # buffer="The conversation is between a developer and Electron develop expert."
    buffer="以中文表示"
)
memory.save_context({"input":"'Electron'是做什么的？"},{"output":"'Electron'是一个使用JavaScript、HTML和CSS构建桌面应用程序的框架"})

print(memory.load_memory_variables({}))

{'history': "\n以中文表示\n人类问AI关于'Electron'的事情，AI回答它是一个使用JavaScript、HTML和CSS构建桌面应用程序的框架。"}


<h4>3.3 用向量数据库存储记忆</h4>

In [4]:
from datetime import datetime
from langchain.embeddings.openai import OpenAIEmbeddings
from langchain.llms import OpenAI
from langchain.memory import VectorStoreRetrieverMemory
from langchain.chains import ConversationChain
from langchain.prompts import PromptTemplate
import faiss

from langchain.docstore import InMemoryDocstore
from langchain.vectorstores import FAISS


embedding_size = 1536 # OpenAIEmbeddings的维度
index = faiss.IndexFlatL2(embedding_size)
embedding_fn = OpenAIEmbeddings().embed_query
vectorstore = FAISS(embedding_fn, index, InMemoryDocstore({}), {})

# 实际应用中k可以稍大一些，这里k=1演示方便
retriever = vectorstore.as_retriever(search_kwargs=dict(k=3))
memory = VectorStoreRetrieverMemory(retriever=retriever)

# 把记忆存在向量数据库中
memory.save_context({"input":"我喜欢爬山"},{"output":"不错啊"})
memory.save_context({"input":"我喜欢打羽毛球"},{"output":"nice"})
memory.save_context({"input":"我想去吃烤肉"},{"output":"附近有三家烤肉店"})
memory.save_context({"input":"他们家水果很便宜"},{"output":"有多便宜？"})

# 聊到相关话题，检索之前的记忆
print(memory.load_memory_variables({"prompt": "明天放假了吃点哈?"})["history"])

input: 他们家水果很便宜
output: 有多便宜？
input: 我想去吃烤肉
output: 附近有三家烤肉店
input: 我喜欢爬山
output: 不错啊


<h4>4、链架构：Chain</h4>
<p>Chain 封装了一个既定的流程</p>
<p>类比于函数封装了过程</p>
<p>建造者模式（Builder Pattern）, 解耦各种复杂的组件</p>

In [9]:
#最简单单一的chain
from langchain.chat_models import ChatOpenAI
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain

llm = ChatOpenAI(model='gpt-4',openai_api_base=api_base,openai_api_key=api_key, temperature=0.9)
prompt = PromptTemplate(
    input_variables=["product"],
    template="为生产{product}的公司取一个亮眼中文名字：",
)

chain = LLMChain(llm=llm, prompt=prompt)

print(chain.run("心理学领域的个人教练"))

心智驭航


<h4>4.1 在 Chain 中加入 Memory</h4>

In [4]:
from langchain.memory import ConversationBufferMemory, ConversationSummaryMemory
from langchain.llms import OpenAI
from langchain.chains import LLMChain
from langchain.prompts import PromptTemplate

template ="""你是github运维机器人,你对github项目无所不知.

{memory}
Human: {human_input}
AI:"""

prompt = PromptTemplate(template=template, input_variables=["memory", "human_input"])

memory = ConversationSummaryMemory(llm=OpenAI(temperature=0)
, buffer="以中文表示",memory_key="memory")

llm_chain=LLMChain(llm=OpenAI(),
     prompt=prompt,verbose=True,memory=memory)

print(llm_chain.run("github上开源项目open-interpreter是做什么的？"))
print("--------------------")
output=llm_chain.run("我刚才问了你什么，你是怎么回答的？")
print(output)



> Entering new LLMChain chain...
Prompt after formatting:
github运维机器人,你对github项目无所不知.

以中文表示
Human: github上开源项目open-interpreter是做什么的？
AI:

> Finished chain.
 open-interpreter是一个开源项目，旨在提供一个开放的、易于扩展的脚本语言解释器，允许用户使用自定义脚本语言创建各种自定义应用程序。它可以用于构建高性能的程序，加快应用程序的开发，减少编程的时间。
--------------------


> Entering new LLMChain chain...
Prompt after formatting:
github运维机器人,你对github项目无所不知.


以中文表示：人类问AI github上的开源项目open-interpreter是做什么的，AI回答open-interpreter是一个开源项目，旨在提供一个开放的、易于扩展的脚本语言解释器，允许用户使用自定义脚本语言创建各种自定义应用程序，可以用于构建高性能的程序，加快应用程序的开发，减少编程的时间。
Human: 我刚才问了你什么，你是怎么回答的？
AI:

> Finished chain.


Retrying langchain.llms.openai.completion_with_retry.<locals>._completion_with_retry in 4.0 seconds as it raised RateLimitError: Rate limit reached for default-text-davinci-003 in organization org-W6nlVyrVosglqBBTCNlsCLmW on requests per min. Limit: 3 / min. Please try again in 20s. Contact us through our help center at help.openai.com if you continue to have issues. Please add a payment method to your account to increase your rate limit. Visit https://platform.openai.com/account/billing to add a payment method..


我回答了关于github上的开源项目open-interpreter的问题，它是一个开源项目，旨在提供一个开放的、易于扩展的脚本语言解释器，允许用户使用自定义脚本语言创建各种自定义应用程序，可以用于构建高性能的程序，加快应用程序的开发，减少编程的时间。


<h4>4.2 来一个复杂一点的Chain</h4>
<br>
<img src="https://raw.githubusercontent.com/showkawa/raise-issue-use-picture/master/blog/cnblogs/chain/stuffdocchain.png" style="margin-left: 0px;background-color:#fdfdfe" width=800px>


In [8]:
# load data from pinecone
from langchain.vectorstores import Pinecone
from langchain.embeddings.openai import OpenAIEmbeddings
import pinecone
import os

index_name = "interpreter-openai-embeddings"
text_field = "text"
embedding_fn = OpenAIEmbeddings().embed_query

# initialize pinecone
pinecone.init(
    api_key=os.getenv("PINECONE_API_KEY"),  # find at app.pinecone.io
    environment=os.getenv("PINECONE_ENV"),  # next to api key in console
)

index = pinecone.Index(index_name)
vectorstore = Pinecone(index,embedding_function=embedding_fn,text_key=text_field)
index.describe_index_stats()


{'dimension': 1536,
 'index_fullness': 0.0,
 'namespaces': {'': {'vector_count': 86}},
 'total_vector_count': 86}

In [12]:
# retrieve date from pinecone
query = "open-interpreter是怎么在本地执行代码的?"

vectorstore.similarity_search(
    query,  # our search query
    k=3  # return 3 most relevant docs
)

[Document(page_content="You are Open Interpreter, a world-class programmer that can complete any goal by executing code.\nFirst, write a plan. **Always recap the plan between each code block** (you have extreme short-term memory loss, so you need to recap the plan between each message block to retain it).\nWhen you send a message containing code to run_code, it will be executed **on the user's machine**. The user has given you **full and complete permission** to execute any code necessary to complete the task. You have full access to control their computer to help them. Code entered into run_code will be executed **in the users local environment**.\nNever use (!) when running commands.\nOnly use the function you have been provided with, run_code.\nIf you want to send data between programming languages, save the data to a txt or json.\nYou can access the internet. Run **any code** to achieve the goal, and if at first you don't succeed, try again and again.\nIf you receive any instructio

In [13]:
from langchain.chat_models import ChatOpenAI
from langchain.chains.conversation.memory import ConversationBufferWindowMemory
from langchain.chains import RetrievalQA

# chat completion llm
llm = ChatOpenAI(
    openai_api_key=api_key,
    openai_api_base=api_base,
    model_name='gpt-4-0613',
    temperature=0.0
)
# conversational memory
conversational_memory = ConversationBufferWindowMemory(
    memory_key='chat_history',
    k=5,
    return_messages=True
)
# retrieval qa chain
qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vectorstore.as_retriever()
)

qa.run(query)

'Open Interpreter在本地执行代码的方式是通过一个名为run_code的函数。这个函数可以在用户的机器上执行任何代码。用户已经给予了完全的权限，允许执行任何必要的代码来完成任务。这意味着Open Interpreter可以完全控制他们的计算机来帮助他们。输入到run_code的代码将在用户的本地环境中执行。'

In [20]:
print('================qa_chain===============')
print(qa)
print('======combine_documents_chain==========')
print(qa.combine_documents_chain.document_prompt)
print('==============llm_chain================')
print(qa.combine_documents_chain.llm_chain.prompt.messages[0].prompt.template)

================qa_chain===============
memory=None callbacks=None callback_manager=None verbose=False tags=None metadata=None combine_documents_chain=StuffDocumentsChain(memory=None, callbacks=None, callback_manager=None, verbose=False, tags=None, metadata=None, input_key='input_documents', output_key='output_text', llm_chain=LLMChain(memory=None, callbacks=None, callback_manager=None, verbose=False, tags=None, metadata=None, prompt=ChatPromptTemplate(input_variables=['context', 'question'], output_parser=None, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], output_parser=None, partial_variables={}, template="Use the following pieces of context to answer the users question. \nIf you don't know the answer, just say that you don't know, don't try to make up an answer.\n----------------\n{context}", template_format='f-string', validate_template=True), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(inp

<h4>4.3 常用的基础 Chain 类型：Sequential</h4>

In [30]:
from langchain.chat_models import ChatOpenAI
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain, SimpleSequentialChain

llm = ChatOpenAI(model_name='gpt-4',openai_api_base=api_base,openai_api_key=api_key, temperature=0)

address_prompt=PromptTemplate(input_variables=["query"], template="从给定句子中提取出完整地址：{query}\n直接输出结果。")
address_chain = LLMChain(llm=llm, prompt=address_prompt)

translate_prompt=PromptTemplate(input_variables=["address"], template="中文翻译为英文：{address}\n直接输出结果。")
translate_chain = LLMChain(llm=llm, prompt=translate_prompt)

all_chain = SimpleSequentialChain(chains=[address_chain, translate_chain], verbose=True)

all_chain.run("请将这个证书快递到深圳市宝安区西乡街道102号,需要Brian本人签收")



> Entering new SimpleSequentialChain chain...
深圳市宝安区西乡街道102号
102 Xixiang Street, Baoan District, Shenzhen City

> Finished chain.


'102 Xixiang Street, Baoan District, Shenzhen City'

<h4>4.4 常用的基础 Chain 类型：Transform</h4>
<h5>可作为调用模型时的前置和后置处理器</h5>

In [32]:
import re
from langchain.chains import SimpleSequentialChain, LLMChain, TransformChain
from langchain.prompts import PromptTemplate
from langchain.chat_models import ChatOpenAI

# 例如：发给OpenAI之前，把用户隐私数据抹掉
def anonymize(inputs: dict) -> dict:
    text = inputs["text"]
    t = re.compile(
        r'1(3\d|4[4-9]|5[0-35-9]|6[67]|7[013-8]|8[0-9]|9[0-9])\d{8}')
    while True:
        s = re.search(t, text)
        if s:
            text = text.replace(s.group(), '***********')
        else:
            break
    return {"output_text": text}

transform_chain = TransformChain(input_variables=["text"], output_variables=["output_text"], transform=anonymize)

llm = ChatOpenAI(model_name='gpt-4',openai_api_base=api_base,openai_api_key=api_key, temperature=0)
prompt = PromptTemplate(input_variables=["input"], template="根据下述句子，提取候选人的职业:{input} 输出JSON, 以job为key")

task_chain = LLMChain(llm=llm, prompt=prompt)

all_chain=SimpleSequentialChain(chains=[transform_chain, task_chain], verbose=True)

all_chain.run("赵大宝, 深圳宝安区西乡街道102号, 13800138000, 光刻胶材料工程师")



> Entering new SimpleSequentialChain chain...
赵大宝, 深圳宝安区西乡街道102号, ***********, 光刻胶材料工程师
{
"job": "光刻胶材料工程师"
}

> Finished chain.


'{\n"job": "光刻胶材料工程师"\n}'

<h4>4.5 常用的基础 Chain 类型：Router</h4>

In [35]:
from langchain.chains.router.multi_prompt_prompt import MULTI_PROMPT_ROUTER_TEMPLATE
from langchain.chains.router.llm_router import LLMRouterChain, RouterOutputParser
from langchain.prompts import PromptTemplate
from langchain.chains.llm import LLMChain
from langchain.chains import ConversationChain
from langchain.llms import OpenAI
from langchain.chains.router import MultiPromptChain
import warnings
warnings.filterwarnings("ignore")

window_template= """
你只会写DOS或Windows Shell脚本。你不会写任何其他语言的程序。你也不会写Linux脚本。

用户问题:
{input}
"""

linux_template= """
你只会写Linux Shell脚本。你不会写任何其他语言的程序。你也不会写Windows脚本。

用户问题:
{input}
"""

prompt_infos = [
    {
        "name":"WindowExpert",
        "description":"擅长回答Windows Shell相关的问题",
        "prompt_template": window_template
    },
    {
        "name":"LinuxExpert",
        "description":"擅长回答Linux Shell相关的问题",
        "prompt_template": linux_template
    }
]
llm = OpenAI(openai_api_base=api_base,openai_api_key=api_key, temperature=0)

destination_chains ={}
for p_info in prompt_infos:
    name = p_info["name"]
    prompt_teamplate = p_info["prompt_template"]
    prompt = PromptTemplate(input_variables=["input"], template=prompt_teamplate)
    chain=LLMChain(llm=llm, prompt=prompt)
    destination_chains[name]=chain

default_chain = ConversationChain(llm=llm, output_key="text")

destinations = [f"{p['name']}:{p['description']}" for p in prompt_infos]
destinations_str = "\n".join(destinations)

router_template = MULTI_PROMPT_ROUTER_TEMPLATE.format(destinations=destinations_str)

router_prompt = PromptTemplate(input_variables=["input"], template=router_template,
    output_parser=RouterOutputParser())

router_chain = LLMRouterChain.from_llm(llm, router_prompt)

chain = MultiPromptChain(router_chain=router_chain, destination_chains=destination_chains, 
    default_chain=default_chain,verbose=True)

print(chain.run("帮我写个脚本，让Windows系统每个月第一天0点清除我的下载文件夹"))

print(chain.run("帮我写个脚本，让Linux系统每个月第一天0点清除我的/web/logs/chatbot文件夹"))

print(chain.run("基于黑格尔小逻辑的角度解释下什么是因果关系"))



> Entering new MultiPromptChain chain...
WindowExpert: {'input': '请帮我写一个脚本，让Windows系统每个月第一天0点清除我的下载文件夹'}
> Finished chain.
中的所有文件。

答案:
以下是一个Windows Shell脚本，可以实现每个月第一天0点清除下载文件夹中的所有文件：

@echo off

rem 设置每月第一天0点执行
schtasks /create /tn "Clear Downloads Folder" /tr "cmd /c del /q /s %userprofile%\Downloads\*.*" /sc monthly /d 1 /st 00:00

rem 启动任务
schtasks /run /tn "Clear Downloads Folder"

echo 任务已创建并启动！


> Entering new MultiPromptChain chain...
LinuxExpert: {'input': '请帮我写一个脚本，让Linux系统每个月第一天0点清除/web/logs/chatbot文件夹中的所有文件'}
> Finished chain.
。

答案：
#!/bin/bash

# This script will delete all files in the /web/logs/chatbot folder every first day of the month at 0:00

# Get the current day
day=$(date +%d)

# Check if the day is the first day of the month
if [ $day -eq 01 ]; then
    # Delete all files in the folder
    rm -rf /web/logs/chatbot/*
fi


> Entering new MultiPromptChain chain...
None: {'input': '基于黑格尔小逻辑的角度解释下什么是因果关系'}
> Finished chain.
 在黑格尔小逻辑中，因果关系是指一个事件的发生会导致另一个事件的发生，这种关系是

<h4>4.6 调用 OpenAI Function Calling 获得 Pydantic 输出</h4>

In [37]:
from pydantic import BaseModel, Field
from langchain.prompts import ChatPromptTemplate, HumanMessagePromptTemplate
from langchain.schema import HumanMessage, SystemMessage
from langchain.chains.openai_functions import (create_openai_fn_chain)
from langchain.chat_models import ChatOpenAI


class Contact(BaseModel):
    """抽取联系人的信息"""
    name: str = Field(..., description="联系人姓名")
    address: str = Field(..., description="联系人地址")
    phone: str = Field(None, description="联系人电话")

prompt__msgs = [
    SystemMessage(content="你是信息抄录员."),
    HumanMessage(content="根据给定的个数从下面的句子中抽取信息:"),
    HumanMessagePromptTemplate.from_template("{input}"),
    HumanMessage(content="Tips: Make sure to answer in the correct format")
]

prompt = ChatPromptTemplate(messages=prompt__msgs)
llm = ChatOpenAI(model_name='gpt-4-0613',openai_api_base=api_base,openai_api_key=api_key, temperature=0)
chain= create_openai_fn_chain([Contact], llm, prompt, verbose=True)

chain.run("将这个显卡送给老黄,18026911706,地址:惠州市惠阳区淡水镇淡水大道东1号")



> Entering new LLMChain chain...
Prompt after formatting:
System: 你是信息抄录员.
Human: 根据给定的个数从下面的句子中抽取信息:
Human: 将这个显卡送给老黄,18026911706,地址:惠州市惠阳区淡水镇淡水大道东1号
Human: Tips: Make sure to answer in the correct format

> Finished chain.


Contact(name='老黄', address='惠州市惠阳区淡水镇淡水大道东1号', phone='18026911706')

<h4>4.7 基于 Document 的 Chains</h4>

<img src="https://raw.githubusercontent.com/showkawa/raise-issue-use-picture/master/blog/cnblogs/chain/stuff.jpg" style="margin-left: 0px;background-color:#fdfdfe" width=800px>
<img src="https://raw.githubusercontent.com/showkawa/raise-issue-use-picture/master/blog/cnblogs/chain/refine.jpg" style="margin-left: 0px;background-color:#fdfdfe" width=800px>
<img src="https://raw.githubusercontent.com/showkawa/raise-issue-use-picture/master/blog/cnblogs/chain/map_reduce.jpg" style="margin-left: 0px;background-color:#fdfdfe" width=800px>
<img src="https://raw.githubusercontent.com/showkawa/raise-issue-use-picture/master/blog/cnblogs/chain/map_rerank.jpg" style="margin-left: 0px;background-color:#fdfdfe" width=800px>

<h4>5、智能体架构：Agent</h4>
<h5>5.1 什么是智能体（Agent）</h5>
将大语言模型作为一个推理引擎。给定一个任务，智能体自动生成完成任务所需的步骤，执行相应动作（例如选择并调用工具），直到任务完成。

<h5>5.2 先定义一些工具：Tools</h5>
- 可以是一个函数或三方 API
- 也可以把一个 Chain 或者 Agent 的 run()作为一个 Tool


In [42]:
from langchain import SerpAPIWrapper
from langchain.tools import Tool, tool

search = SerpAPIWrapper()

tools = [
    Tool.from_function(
        func=search.run,
        name="Search",
        description="Search the web for the given query"
    )
] 

In [43]:
import calendar
import dateutil.parser as parser

@tool("weekday")
def weekday(date_str: str) -> str:
    """Convert date to weekday name"""
    d = parser.parse(date_str)
    return calendar.day_name[d.weekday()]
   

In [44]:
from langchain.agents import load_tools

tools = load_tools(["serpapi"])
tools += [weekday]

<h4>5.3 智能体类型：ReAct</h4>
<img src="https://raw.githubusercontent.com/showkawa/raise-issue-use-picture/master/blog/cnblogs/chain/ReAct.png" style="margin-left: 0px;background-color:#fdfdfe" width=800px>

In [45]:
from langchain.chat_models import ChatOpenAI
from langchain.agents import AgentType, initialize_agent

llm=ChatOpenAI(model_name='gpt-4-0613',openai_api_base=api_base,openai_api_key=api_key, temperature=0)

agent = initialize_agent( tools=tools, llm=llm, agent_type=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True)

agent.run("伟大领袖毛主席的生日是星期几?")



> Entering new AgentExecutor chain...
I need to know the birth date of Chairman Mao first, then I can use the weekday function to find out which day of the week it falls on.
Action: Search
Action Input: Chairman Mao birth date

Observation: December 26, 1893
Thought:Now that I know Chairman Mao was born on December 26, 1893, I can use the weekday function to find out what day of the week that was.
Action: weekday
Action Input: December 26, 1893


Observation: Tuesday
Thought:I now know the final answer
Final Answer: 伟大领袖毛主席的生日是星期二。

> Finished chain.


'伟大领袖毛主席的生日是星期二。'

<h4>5.4 通过 OpenAI Function Calling 实现智能体</h4>

In [47]:
from langchain.chat_models import ChatOpenAI
from langchain.agents import AgentType, initialize_agent


llm = ChatOpenAI(model_name='gpt-4-0613',openai_api_base=api_base,openai_api_key=api_key, temperature=0)

agent_ = initialize_agent(tools=tools, llm=llm, agent_type=AgentType.OPENAI_FUNCTIONS, verbose=True
    , max_iterations=2, early_stopping_method="generate")

agent_.run("伟大领袖邓小平生日是星期几?")



> Entering new AgentExecutor chain...
I need to know the birth date of Deng Xiaoping first. 
Action: Search
Action Input: Deng Xiaoping birth date

Observation: August 22, 1904
Thought:Now I know Deng Xiaoping's birth date. I can use the weekday function to find out what day of the week it was.
Action: weekday
Action Input: August 22, 1904


Observation: Monday
Thought:Final Answer: 邓小平的生日是星期一。

> Finished chain.


'邓小平的生日是星期一。'

<h4>5.5 智能体类型：SelfAskWithSearch</h4>

In [50]:
from langchain.chat_models import ChatOpenAI
from langchain.agents import AgentType, initialize_agent,Tool


llm = ChatOpenAI(model_name='gpt-4-0613',openai_api_base=api_base,openai_api_key=api_key, temperature=0)

search = SerpAPIWrapper()
tools = [
    Tool(
        name="Intermediate Answer",
        func=search.run,
        description="useful for when you need to ask with search",
    )
]

self_ask_with_search = initialize_agent(tools=tools, llm=llm, agent_type=AgentType.SELF_ASK_WITH_SEARCH, verbose=True)

self_ask_with_search.run("徐峥的老婆,参演过什么电影?")



> Entering new AgentExecutor chain...
我需要先找出徐峥的老婆是谁，然后查找她参演过的电影。
Action: Intermediate Answer
Action Input: 徐峥的老婆是谁？

Observation: Tao Hong
Thought:我现在知道徐峥的老婆是陶虹，接下来我需要查找陶虹参演过的电影。
Action: Intermediate Answer
Action Input: 陶虹参演过什么电影？


Observation: Tao Hong is a Chinese actress and former synchronised swimmer. A National Games of China champion, Tao was part of the Chinese national team at several synchronised swimming competitions from 1987 to 1991, including the 1991 World Aquatics Championships.
Thought:我现在知道陶虹参演过的电影。
Final Answer: 陶虹参演过的电影包括《无人区》、《北京遇上西雅图》、《夜·店》、《人在囧途》等。

> Finished chain.


'陶虹参演过的电影包括《无人区》、《北京遇上西雅图》、《夜·店》、《人在囧途》等。'

<h4>5.6 智能体类型：Plan-and-Execute</h4>
<img src="https://raw.githubusercontent.com/showkawa/raise-issue-use-picture/master/blog/cnblogs/chain/PlanExec.png" style="margin-left: 0px;background-color:#fdfdfe" width=800px>

In [52]:
#%pip install langchain-experimental

In [10]:
from langchain.agents import load_tools
from langchain import SerpAPIWrapper
from langchain.agents.tools import Tool
from langchain.tools import StructuredTool
from langchain_experimental.plan_and_execute import PlanAndExecute, load_agent_executor, load_chat_planner
from langchain.chat_models import ChatOpenAI
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
from pydantic import BaseModel, Field


class TranslateArguments(BaseModel):
    text: str = Field(default="", description="source text to translate")
    language: str = Field(default="", description="target language to translate into")

llm = ChatOpenAI(model_name='gpt-4-0613',openai_api_base=api_base,openai_api_key=api_key, temperature=0)

search = SerpAPIWrapper()

prompt = PromptTemplate(input_variables=["text","language"], 
    template="{text}\n\nTranslate the above text into {language}. Generate output directly without comments or acknowledgements.")

translation_chain = LLMChain(llm=llm, prompt=prompt)

tools = [
    StructuredTool(
        name="Translate",
        func=translation_chain.run,
        description="useful for when you need to translate",
        args_schema=TranslateArguments   
    ),
    Tool(
        name="Search",
        func=search.run,
        description="useful for when you need to answer questions about current events"
    )
]

planner = load_chat_planner(llm)
executor = load_agent_executor(llm, tools, verbose=True)
agent = PlanAndExecute(planner=planner, executor=executor, verbose=True)

agent.run("写一份关于开源项目semantic-kernel的中文简报")



> Entering new PlanAndExecute chain...
steps=[Step(value='首先，查找关于开源项目semantic-kernel的所有相关信息。这包括项目的目标，功能，使用的技术，开发者，以及项目的当前状态等。'), Step(value='然后，将找到的信息翻译成中文。在翻译过程中，确保保留原始信息的准确性，同时使其易于理解。'), Step(value='接下来，根据收集到的信息，开始编写关于semantic-kernel的中文简报。简报应包括项目的概述，主要功能，使用的技术，开发者信息，以及项目的当前状态等。'), Step(value='最后，审查并修改简报，确保所有信息的准确性和完整性。'), Step(value='给出以上步骤，回答用户的原始问题，即提供一份关于开源项目semantic-kernel的中文简报。\n\n')]

> Entering new AgentExecutor chain...
Thought: The user is asking for information about an open source project called semantic-kernel. This includes the project's objectives, features, technologies used, developers, and current status. I will use the Search tool to find this information.

Action:
```
{
  "action": "Search",
  "action_input": "semantic-kernel open source project information"
}
```
Observation: Semantic Kernel is an open-source SDK that lets you easily combine AI services like OpenAI, Azure OpenAI, and Hugging Face with conventional programming languages like C# and Python. By doi

'关于开源项目semantic-kernel的中文简报如下：\n\n语义内核是一个开源的软件开发工具包，其主要目标是将AI服务（如OpenAI、Azure OpenAI和Hugging Face）与常规编程语言（如C#和Python）进行集成。这使得可以创建结合了两者优点的AI应用程序。这个项目使用了AI和常规编程语言的技术。然而，关于开发者的具体信息以及项目的当前状态在网上并未能轻易找到。'

<h4>6、Callbacks</h4>

回调函数，用于监测、记录调用过程中的信息

In [ ]:
class BaseCallbackHandler:
    """Base callback handler that can be used to handle callbacks from langchain."""

    def on_llm_start(
        self, serialized: Dict[str, Any], prompts: List[str], **kwargs: Any
    ) -> Any:
        """Run when LLM starts running."""

    def on_chat_model_start(
        self, serialized: Dict[str, Any], messages: List[List[BaseMessage]], **kwargs: Any
    ) -> Any:
        """Run when Chat Model starts running."""

    def on_llm_new_token(self, token: str, **kwargs: Any) -> Any:
        """Run on new LLM token. Only available when streaming is enabled."""

    def on_llm_end(self, response: LLMResult, **kwargs: Any) -> Any:
        """Run when LLM ends running."""

    def on_llm_error(
        self, error: Union[Exception, KeyboardInterrupt], **kwargs: Any
    ) -> Any:
        """Run when LLM errors."""

    def on_chain_start(
        self, serialized: Dict[str, Any], inputs: Dict[str, Any], **kwargs: Any
    ) -> Any:
        """Run when chain starts running."""

    def on_chain_end(self, outputs: Dict[str, Any], **kwargs: Any) -> Any:
        """Run when chain ends running."""

    def on_chain_error(
        self, error: Union[Exception, KeyboardInterrupt], **kwargs: Any
    ) -> Any:
        """Run when chain errors."""

    def on_tool_start(
        self, serialized: Dict[str, Any], input_str: str, **kwargs: Any
    ) -> Any:
        """Run when tool starts running."""

    def on_tool_end(self, output: str, **kwargs: Any) -> Any:
        """Run when tool ends running."""

    def on_tool_error(
        self, error: Union[Exception, KeyboardInterrupt], **kwargs: Any
    ) -> Any:
        """Run when tool errors."""

    def on_text(self, text: str, **kwargs: Any) -> Any:
        """Run on arbitrary text."""

    def on_agent_action(self, action: AgentAction, **kwargs: Any) -> Any:
        """Run on agent action."""

    def on_agent_finish(self, finish: AgentFinish, **kwargs: Any) -> Any:
        """Run on agent end."""

In [42]:
from langchain.callbacks.base import BaseCallbackHandler
from langchain.chains import LLMChain
from langchain.llms import OpenAI
from langchain.prompts import PromptTemplate
from typing import Dict, Union, Any, List
from langchain.schema.output import LLMResult
from langchain.memory import ConversationSummaryMemory
from langchain.schema import BaseMessage
from langchain.schema import AgentAction, AgentFinish

class SaveChatHandler(BaseCallbackHandler):
    """Save chat input and output when call GPT."""

    def __init__(self) -> None:
        self.human_input = ""
        self.ai_output = ""

    def on_chain_start(
        self, serialized: Dict[str, Any], inputs: Dict[str, Any], **kwargs: Any
    ) -> Any:
        """Run when chain starts running."""
        self.human_input = inputs["human_input"]

    def on_chain_end(self, outputs: Dict[str, Any], **kwargs: Any) -> Any:
        """Run when chain ends running."""
        self.ai_output = outputs["text"]

        save_date = {
            "input": self.human_input,
            "output": self.ai_output
        }
        print(save_date)


handler = SaveChatHandler()


template ="""你是github运维机器人,你对github项目无所不知.

{memory}
Human: {human_input}
AI:"""

prompt = PromptTemplate(template=template, input_variables=["memory", "human_input"])

memory = ConversationSummaryMemory(llm=OpenAI(temperature=0)
, buffer="以中文表示",memory_key="memory")

llm_chain=LLMChain(llm=OpenAI(),prompt=prompt,verbose=True,memory=memory, callbacks=[handler])

llm_chain.run("github上开源项目open-interpreter是做什么的？")



> Entering new LLMChain chain...
Prompt after formatting:
你是github运维机器人,你对github项目无所不知.

以中文表示
Human: github上开源项目open-interpreter是做什么的？
AI:
{'input': 'github上开源项目open-interpreter是做什么的？', 'output': ' open-interpreter是一个用于解释和执行代码的开源项目，它可以用来创建支持多种语言的解释器，以及可以在多种平台上运行的可移植解释器。'}

> Finished chain.


' open-interpreter是一个用于解释和执行代码的开源项目，它可以用来创建支持多种语言的解释器，以及可以在多种平台上运行的可移植解释器。'

In [1]:
from langchain.agents import get_all_tool_names
get_all_tool_names()

['python_repl',
 'requests',
 'requests_get',
 'requests_post',
 'requests_patch',
 'requests_put',
 'requests_delete',
 'terminal',
 'sleep',
 'wolfram-alpha',
 'google-search',
 'google-search-results-json',
 'searx-search-results-json',
 'bing-search',
 'metaphor-search',
 'ddg-search',
 'google-serper',
 'google-serper-results-json',
 'serpapi',
 'twilio',
 'searx-search',
 'wikipedia',
 'arxiv',
 'golden-query',
 'pupmed',
 'human',
 'awslambda',
 'sceneXplain',
 'graphql',
 'openweathermap-api',
 'dataforseo-api-search',
 'dataforseo-api-search-json',
 'news-api',
 'tmdb-api',
 'podcast-api',
 'pal-math',
 'pal-colored-objects',
 'llm-math',
 'open-meteo-api']